# 02 - Data Preprocessing & Feature Preparation

## Objective
Notebook này thực hiện các bước tiền xử lý dữ liệu cho bài toán phân tích hiệu suất và nghỉ việc của nhân viên, bao gồm:
- Khảo sát và làm sạch dữ liệu
- Xử lý giá trị thiếu
- Loại bỏ các cột không mang thông tin
- Mã hóa biến phân loại
- Chuẩn hóa dữ liệu số
- Chuẩn bị dữ liệu cho các bước khai phá và mô hình hóa tiếp theo

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

In [ ]:
DATA_PATH = "../data/raw/HR_Analytics.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Kiểm tra kích thước dữ liệu
print("Shape:", df.shape)

In [ ]:
# Kiểm tra missing values
df.isnull().sum()

In [ ]:
# Xử lý missing values
df['YearsWithCurrManager'].fillna(
    df['YearsWithCurrManager'].median(), inplace=True
)

df.isnull().sum()

In [ ]:
# Loại bỏ các cột không có giá trị phân tích
drop_cols = [
    'EmpID',
    'EmployeeNumber',
    'EmployeeCount',
    'Over18',
    'StandardHours'
]

df.drop(columns=drop_cols, inplace=True)
df.shape

In [ ]:
# Phân tách biến mục tiêu (Target)
target_col = 'Attrition'

y = df[target_col]
X = df.drop(columns=[target_col])

In [ ]:
# Mã hóa biến mục tiêu
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

pd.Series(y_encoded).value_counts()

In [ ]:
# Xác định biến phân loại & biến số
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

categorical_cols, numerical_cols

In [ ]:
# One-Hot Encoding cho biến phân loại
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

X_encoded.shape

In [ ]:
# Chuẩn hóa biến số
scaler = StandardScaler()

X_scaled = X_encoded.copy()
X_scaled[numerical_cols] = scaler.fit_transform(
    X_scaled[numerical_cols]
)

X_scaled.describe()

In [ ]:
# Gộp lại dữ liệu hoàn chỉnh
df_processed = X_encoded.copy()
df_processed['Attrition'] = y_encoded

df_processed.head()

In [ ]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== File cho Apriori =====
df_processed.to_csv(
    os.path.join(OUTPUT_DIR, "hr_processed.csv"),
    index=False
)
print("Saved Apriori file: hr_processed.csv")

# ===== File cho ML / Clustering =====
df_ml = X_scaled.copy()
df_ml["Attrition"] = y_encoded

df_ml.to_csv(
    os.path.join(OUTPUT_DIR, "hr_processed_ml.csv"),
    index=False
)
print("Saved ML file: hr_processed_ml.csv")

## Summary

Sau bước preprocessing:
- Dữ liệu không còn missing values
- Các cột không mang giá trị phân tích đã được loại bỏ
- Biến phân loại đã được mã hóa
- Biến số đã được chuẩn hóa
- Dữ liệu sẵn sàng cho các bước:
  - Khai phá luật kết hợp
  - Phân cụm
  - Phân lớp & bán giám sát